In [ ]:
import re
from pathlib import Path
import pandas as pd
from rapidfuzz import process, fuzz
from unidecode import unidecode
from tqdm import tqdm
import yaml

import glob
import warnings

In [ ]:
import os
os.chdir('../../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

## Merge all files from Qichacha

### (1) name link

In [ ]:
# Suppress openpyxl warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

folder = os.path.join(
    dataset_config['path_csmar'],
    'CSMAR_subs_Engname',
    'Qichacha_name_link'
)

file_list = glob.glob(os.path.join(folder, '*.xlsx'))
file_list.sort()

if not file_list:
    raise FileNotFoundError(f"No .xlsx files found in: {folder}")

dfs = []

for f in file_list:
    # 1) read without header
    df = pd.read_excel(f, header=None)
    
    # 2) drop the first useless row
    df = df.iloc[1:, :].reset_index(drop=True)
    
    # 3) use the new first row as column names
    df.columns = df.iloc[0]
    
    # 4) data starts from the next row
    df = df.iloc[1:, :].reset_index(drop=True)

    # 5) keep only rows with 匹配结果 != "失败匹配"
    required_cols = {"匹配结果", "导入名称", "匹配企业名称"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"Missing required columns in file: {f}")

    df = df[df["匹配结果"] != "失败匹配"]

    # 6) only keep 导入名称 and 匹配企业名称
    df = df[["导入名称", "匹配企业名称"]]

    dfs.append(df)

# Final concatenated DataFrame
Qichacha_name_link = pd.concat(dfs, ignore_index=True, sort=False).drop_duplicates()

Qichacha_name_link

In [ ]:
Qichacha_name_link.rename(columns={'匹配企业名称': '企业名称'}, inplace=True)

In [ ]:
Qichacha_name_link.to_csv(dataset_config['path_csmar'] + 'CSMAR_subs_Engname/Qichacha_name_link.csv', index=False)

### (2) Mathced info

In [ ]:
# Suppress openpyxl 'no default style' warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

folder = os.path.join(
    dataset_config['path_csmar'],
    'CSMAR_subs_Engname',
    'Qichacha_firm_info'
)

file_list = glob.glob(os.path.join(folder, '*.xlsx'))
file_list.sort()

if not file_list:
    raise FileNotFoundError(f"No .xlsx files found in: {folder}")

dfs = []

for f in file_list:
    # (1) read without header and delete the first (useless) row
    df_raw = pd.read_excel(f, header=None)
    df_no_first = df_raw.iloc[1:, :].reset_index(drop=True)

    # (2) new first row is variable names
    header = df_no_first.iloc[0]
    df_data = df_no_first.iloc[1:, :].reset_index(drop=True)
    df_data.columns = header

    # check required columns
    required_cols = {"企业名称", "英文名"}
    if not required_cols.issubset(df_data.columns):
        raise ValueError(f"Missing required columns in file: {f}")

    # (3) keep only 企业名称 and 英文名; no row is dropped here
    df_data = df_data[["企业名称", "英文名"]]

    dfs.append(df_data)

# Final concatenated DataFrame (all files, all lines kept)
Qichacha_firm_info = pd.concat(dfs, ignore_index=True, sort=False).drop_duplicates()

Qichacha_firm_info

In [ ]:
Qichacha_firm_info = Qichacha_firm_info.drop_duplicates(subset='企业名称', keep='first')
Qichacha_firm_info

In [ ]:
Qichacha_firm_info.to_csv(dataset_config['path_csmar'] + 'CSMAR_subs_Engname/Qichacha_firm_info.csv', index=False)

## Merge to one file

In [ ]:
Qichacha_eng = pd.merge(Qichacha_name_link, Qichacha_firm_info, on='企业名称', how='left')
Qichacha_eng.rename(columns={'导入名称': 'CSMAR_subs_name', '企业名称': 'Qichacha_matched_name', '英文名': 'Qichacha_eng_name'}, inplace=True)
Qichacha_eng

In [ ]:
Qichacha_eng.to_csv(dataset_config['path_processed'] + 'WOS_listed/Qichacha_firm_subs_eng.csv', index=False)